# 🗂️ 3. Vector & Graph RAG: 비정형 문서 인덱싱·하이브리드 검색과 지식 그래프

앞선 `2_Agentic_RAG.ipynb`에서 다룬 에이전트가 정확한 답변을 생성하려면, 검색 도구가 참조하는 **데이터베이스의 인덱싱 품질**이 뒷받침되어야 합니다.
아무리 뛰어난 추론 모델을 사용하더라도, 문서를 자르는 과정에서 문맥이 유실되었거나 문서 간 연결 고리가 끊어져 있다면 에이전트는 올바른 근거를 찾을 수 없습니다.
실무 RAG 시스템에서 검색 품질을 좌우하는 가장 큰 요인은 모델 자체가 아니라 **문서 전처리(파싱)와 인덱싱 파이프라인 설계**에 있습니다.

---

### 🎯 학습 목표
1. 문서 인덱싱 과정에서 발생하는 두 가지 대표적 문제(**문맥 유실, 포맷 파편화**)와 이를 전처리 단계에서 해결하는 **표준 포맷 정규화** 방법론을 이해합니다.
2. 계층 태그가 없는 비정형 PDF에서 구조 정보(북마크, 반복 패턴)를 추출해 표준 Markdown 계층(`H1~H4`)으로 복원하고, **문맥 헤더(`search` / `read`)와 도메인 메타데이터**를 결합해 문서 형식에 상관없이 단일 파이프라인으로 인덱싱합니다.
3. **사내 용어 사전, 하이브리드 검색(Dense + BM25), 윈도우 확장(Window Expansion)** 을 결합하여 약어, 조항 번호, 잘린 표까지 정확히 찾아내는 검색기를 구현합니다.
4. 단순 벡터 검색으로 답하기 어려운 **복합 관계(Multi-Hop) 질문과 거시적 요약 질문**을 지식 그래프 및 **Microsoft GraphRAG**로 해결하는 원리를 익힙니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 기법 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | 프로젝트 경로 및 모델 초기화 |
| **Part 1** | 인덱싱의 두 가지 핵심 문제 | 문맥 유실과 포맷 파편화 → 표준 포맷 정규화 |
| **Part 2** | 문서 정규화: 표준 Markdown 변환 | 계층 구조 매핑, 북마크·패턴 기반 PDF 정규화 및 품질 검수 |
| **Part 3** | 문맥 보존 청킹: 작게 찾고 크게 읽기 ① | 검색용/읽기용 문맥 헤더 분리 (`toc={"search": 2, "read": 3}`, Breadcrumb) |
| **Part 4** | 단일 파이프라인 통합 인덱싱 | `index_documents()`, 도메인 메타데이터 태깅 및 필터링 |
| **Part 5** | 하이브리드 검색: 의미와 키워드 결합 | 도메인 용어 사전, BM25 + Dense 검색 융합 (RRF) |
| **Part 6** | 문맥 복원: 작게 찾고 크게 읽기 ② | 인접 청크 결합을 통한 윈도우 확장 (`ContextualWindow`) |
| **Part 7** | 지식 그래프 기초 (직접 구축) | 도메인 온톨로지 기반 트리플렛 추출, NetworkX 다중 홉 탐색 |
| **Part 8** | Microsoft GraphRAG | 커뮤니티 요약 리포트, Local/Global Search, 에이전트용 그래프 탐색 도구 |
| **Part 9** | 정리 및 운영 DB 검증 | 파이프라인 종합 정리, 사전 구축 DB 통합 검증 및 신규 규정 확장 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)

print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 공통 유틸리티 로드
from app.utils import init_chat_model, normalize_content

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)

---
## 🧩 Part 1. 인덱싱의 두 가지 핵심 문제

문서를 검색 가능한 형태로 데이터베이스에 적재(인덱싱)할 때는 긴 문서를 일정한 길이의 작은 조각(청크, Chunk)으로 나누어 저장합니다. 하지만 현업 문서를 별도의 구조 처리 없이 글자 수 기준으로만 잘라 넣으면 다음 두 가지 문제가 발생합니다.

### 문제 1. 청킹 과정에서 발생하는 문맥 유실 (탈맥락화, Decontextualization)

문서는 보통 **`문서 제목(H1) ➔ 장(H2) ➔ 조항(H3) ➔ 기준표 · 세부 항 · 단서`** 순의 계층 구조를 가집니다. 문서 제목과 장·조항 제목은 문서나 절의 **맨 위에 한 번만 등장**하지만, 사람이 읽을 때는 이 상위 제목을 머릿속에 유지한 채 아래쪽의 세부 항목들을 해석합니다. 즉, **상위 제목이 하위 문단 전체의 전역적 맥락(Global Context) 역할**을 합니다.

문제는 문서를 고정 길이(예: 500자)로 위에서부터 단순 분할할 때 발생합니다. **상단에 한 번만 적힌 문서 제목과 조항 헤더는 앞쪽 청크에만 남고, 글자 수 경계 아래에서 잘려 나온 하위 청크에는 말단 본문 문장만 단독으로 저장**됩니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    subgraph ORIG["1. 원본 문서 (상위 헤더의 맥락이 아래 항까지 상속됨)"]
        direction TB
        H_ALL["[문서 상단 헤더 — 1회만 등장]<br/># 국내 여비 지급 규정 (H1)<br/>## 제3장 숙박비·일비·식비 (H2)<br/>### 제15조 일비·식비 지급 기준 (H3)"]
        P_TBL["[제15조 1항 · 기준표]<br/>직급별 1일 정액: 팀장 3만 / 팀원 2.5만"]
        P_SUB["[제15조 2항 · 하위 문단]<br/>'당일 출장은 식비 12,000원,<br/>일비는 기준액의 50%를 지급한다.'"]
        H_ALL -->|"전역 맥락 상속<br/>(국내 출장 · 제15조)"| P_TBL
        P_TBL -->|"기준액(2.5만) 참조"| P_SUB
    end

    ORIG ==>|"고정 길이(500자)<br/>단순 분할 적재"| VDB

    subgraph VDB["2. Vector DB 저장 상태 (하위 청크가 앞뒤 맥락 없이 고립됨)"]
        direction TB
        C14["앞 청크 (Chunk 14)<br/># 국내 여비 규정 > 제3장 > 제15조<br/>1항 기준표: 팀장 3만 / 팀원 2.5만"]
        C15["검색된 하위 청크 (Chunk 15)<br/>'2. 당일 출장은 식비 12,000원,<br/>일비는 기준액의 50%를 지급한다.'"]
        LOSS["❌ 하위 청크 단독 조회 시 발생하는 탈맥락화<br/>① 전역 맥락 소실: 상위 헤더가 앞 청크에만 남아 국내/해외·소속 조항 불명<br/>② 인접 맥락 단절: 직전 청크의 기준표(2.5만)와 분리되어 50% 금액 산출 불가"]
        C14 -.-x|"✂️ 500자 경계 절단<br/>(헤더·기준표는 앞 청크에만 잔류)"| C15
        C15 --> LOSS
    end

    style H_ALL fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style P_TBL fill:#f5f5f5,stroke:#9e9e9e,stroke-width:1px
    style P_SUB fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style C14 fill:#f5f5f5,stroke:#9e9e9e,stroke-width:1px
    style C15 fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style LOSS fill:#ffebee,stroke:#d32f2f,stroke-width:1.5px
```

그 결과 Vector DB에서 하위 청크(`Chunk 15`)가 검색되더라도 두 가지 단절이 동시에 일어납니다.
1. **전역·상위 맥락 소실 (수직적 단절)**: 상위 경로(`국내 여비 규정 > 제3장 > 제15조`)가 청크 텍스트에 없으므로, 국내 출장인지 해외 출장인지조차 구별할 수 없습니다. 특히 여러 권의 기술 백서마다 `개요(Overview)`, `안티패턴(Common anti-patterns)`처럼 동일한 소제목이 반복될 때는 **잘린 청크만 보고 보안 백서의 내용인지 비용 최적화 백서의 내용인지 판별할 수 없습니다.**
2. **인접 문맥 절단 (수평적 단절)**: 글자 수 경계에서 앞 청크의 정액 기준표(`팀원 25,000원`)와 뒤 청크의 감액 조건(`기준액의 50%`)이 갈라져, 뒤 청크만 읽고는 실제 지급 금액을 계산할 수 없습니다.

---

### 문제 2. 문서 포맷 파편화와 해법: 표준 포맷 정규화

기업 내부 문서는 깔끔한 Markdown(`.md`)뿐 아니라 수백 페이지 분량의 PDF 백서(`.pdf`), 워드(`.docx`), 텍스트(`.txt`) 등 다양한 형식으로 흩어져 있습니다. 문서 형식마다 별도의 청킹·인덱싱 파이프라인을 만들면 코드가 중복되고 유지보수가 어려워집니다.

해결책은 청킹 전에 **모든 형식의 문서를 `H1~H4` 계층 구조를 갖춘 표준 Markdown 규격으로 먼저 통일(정규화)** 하는 것입니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    subgraph RAW["다양한 원본 문서 포맷"]
        direction TB
        A1["사내 규정 (.md)<br/>표준 헤더 보유"]
        A2["기술 백서 (.pdf)<br/>계층 태그 없음 (평문·북마크)"]
        A3["신규 양식 문서<br/>(규칙 프리셋만 추가)"]
    end

    NORM["포맷별 정규화기<br/>(to_markdown)<br/>북마크 · 패턴 기반 계층 복원"]
    STD["표준 Markdown 규격 통일<br/># H1 문서명<br/>## H2 대분류 장<br/>### H3 핵심 조항 / #### H4 세부 항목"]
    CHUNK["단일 문맥 청킹기<br/>(ContextualHeader)<br/>상위 경로 · 목차 자동 주입"]
    RETR["통합 하이브리드 검색기<br/>(Dense + BM25 + 윈도우 확장)"]

    A1 --> NORM
    A2 --> NORM
    A3 --> NORM
    NORM ==> STD ==> CHUNK ==> RETR

    style NORM fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style STD fill:#fff8e1,stroke:#f9a825,stroke-width:2px
    style CHUNK fill:#ede7f6,stroke:#5e35b1,stroke-width:1.5px
    style RETR fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

| 인덱싱 문제 | 표준 포맷 정규화를 통한 해결 방식 |
|:---|:---|
| **포맷 파편화** | 원본 파일 형식이 무엇이든 `H1~H4` 표준 Markdown으로 먼저 통일하므로, 이후의 청킹·인덱싱·검색 코드를 **단일 파이프라인으로 공유**합니다. |
| **문맥 유실** | 정규화된 Markdown에 명시된 `문서명 > 장 > 조항 > 세부 항목` 계층을 읽어 **모든 하위 청크 상단에 상위 맥락 헤더를 자동 부착**합니다. |

### 🧭 이번 실습에서 다루는 전체 파이프라인

| 단계 | 핵심 기법 | 해결하려는 문제 |
|:---:|:---|:---|
| **1. 정규화 (Part 2)** | 북마크·패턴 기반 `H1~H4` 계층 복원 및 보존율 검수 | 계층 태그가 없는 PDF에서 문서 구조가 깨지거나 텍스트가 누락되는 문제 |
| **2. 청킹 (Part 3~4)** | 검색용(`search`)·읽기용(`read`) 문맥 헤더 분리 및 메타데이터 태깅 | 하위 청크의 전역 맥락 소실 및 긴 목차로 인한 임베딩 희석 문제 |
| **3. 검색 (Part 5)** | 사내 용어 사전 + BM25 & Dense 하이브리드 검색 (RRF) | 사내 약어, 조항 번호, 고유명사를 순수 벡터 검색만으로 찾지 못하는 문제 |
| **4. 복원 (Part 6)** | 인접 청크 윈도우 확장 (`ContextualWindow`) | 청크 경계에서 기준표나 단서 조항이 앞뒤로 잘려 나가는 문제 |

---
## 📑 Part 2. 문서 정규화: 모든 문서를 동일한 표준 Markdown 규격으로

### 정규화의 목표: 표준 Markdown 4단계 계층 규격 (`H1~H4`)

인덱싱에서 말하는 정규화란 단순히 "PDF에서 글자를 추출하는 작업"이 아니라, **서로 다른 형식의 문서들을 하나의 약속된 계층 구조(`H1~H4`)로 통일하는 작업**입니다. 이 실습에서 사용하는 표준 마크다운 계층 규격은 다음과 같습니다.

| 규격 요소 | 계층 의미 | 적용 예시 1: 사내 규정 (`.md`) | 적용 예시 2: 기술 아키텍처 백서 (`.pdf`) |
|:---|:---|:---|:---|
| **H1 (`#`)** | 문서 최상위 제목 | 규정명 (`# 국내 여비 및 교통비 지급 규정`) | 백서 제목 (`# Performance Efficiency Pillar - ...`) |
| **H2 (`##`)** | 대분류 장 (Chapter) | 장 (`## 제3장 숙박비, 일비 및 식비`) | 아키텍처 영역 (`## Selection`, `## Compute`, ...) |
| **H3 (`###`)** | 핵심 조항 / 세부 주제 | 조항 (`### 제15조 일비 및 식비 기준`) | 모범 사례 (`### PERF02-BP01 Select the best compute...`) |
| **H4 (`####`)** | 조항 내부 상세 항목 | 세부 항·호 | `#### Desired outcome`, `#### Common anti-patterns`, `#### Implementation steps` |
| **메타데이터** | 문서 분류 속성 | 출처 파일명 | 출처 파일명, 아키텍처 영역(`pillar`) |

전처리 단계에서 이 4단계 규격만 맞춰 두면, Part 3 이후에 수행할 청킹, 문맥 헤더 부착(`search: H1~H2` / `read: H1~H3`), 인덱싱, 검색 과정은 **원본 문서가 어떤 파일 형식이었는지 전혀 신경 쓸 필요가 없습니다.**

---

### 2.1 마크다운 문서 (.md): 이미 갖춰진 계층 구조 확인

처음부터 Markdown 형식으로 작성된 문서는 문서 제목(`#`), 장(`##`), 조항(`###`)이 태그로 명시되어 있어 별도의 구조 복원 작업이 필요하지 않습니다.
예시 데이터인 사내 규정 5종(`여비교통비`, `인사복무`, `정보보안`, `전사위임전결`, `클라우드및GPU인프라`)을 열어 마크다운 헤더 계층이 어떻게 구성되어 있는지 확인해 보겠습니다.

In [ ]:
import glob
import os

policy_files = sorted(glob.glob("data/policies/*.md"))
print(f"📂 발견된 사내 규정 파일 ({len(policy_files)}개):")
for f in policy_files:
    print(" -", os.path.basename(f))

# 첫 번째 규정(여비지급규정) 원문의 헤더 계층 확인
with open(policy_files[0], "r", encoding="utf-8") as f:
    sample_policy_text = f.read()

headers = [ln for ln in sample_policy_text.splitlines() if ln.startswith("#")]
print(f"\n📌 {os.path.basename(policy_files[0])}의 마크다운 계층 구조:")
for h in headers[:8]:
    print(" ", h)

💡 이처럼 마크다운 원문은 별도의 파싱 규칙 없이도 `# 규정명` ➜ `## 장` ➜ `### 조항`의 표준 계층을 그대로 활용할 수 있습니다.

---

### 2.2 동일한 형식의 대량 문서 인덱싱 (.pdf): 북마크와 패턴을 활용한 계층 복원

실무에서 가장 까다로운 대상은 수백 페이지에 달하는 **비정형 PDF 문서**입니다.
PDF 파일은 화면과 인쇄물에서 보이는 시각적 배치(글자 크기, 굵기, 좌표)만 저장할 뿐, "이 줄은 2단계 장 제목(`H2`)이고 이 줄은 3단계 소제목(`H3`)이다"라는 의미론적 계층 정보를 직접 담고 있지 않는 경우가 많습니다.

이 때문에 일반적인 자동화 PDF 파서(예: `pymupdf4llm` 기본 설정)로 문서를 변환하면 다음과 같은 문제가 흔히 발생합니다.
1. **글자 크기에만 의존한 부정확한 헤더 추정**: 본문 중 강조된 굵은 글씨나 표 제목을 상위 헤더로 오인하거나, 두 줄로 넘어간 긴 제목을 불완전하게 쪼갭니다.
2. **하위 소제목 계층 유실**: 본문과 글자 크기가 비슷하지만 굵은 글씨로 구분된 내부 소제목들은 마크다운 헤더(`###`, `####`)로 인식되지 못하고 일반 평문(`**...**`)으로 묻혀 버립니다.
3. **페이지 노이즈 혼입**: 매 페이지 상단·하단에 반복되는 머리말(Running Header)이나 목차의 점선(`......`)이 본문 텍스트 중간에 그대로 섞여 들어갑니다.

하지만 기업에서 다루는 대량의 PDF 문서(기술 백서 시리즈, 제품 매뉴얼, 정기 보고서 등)는 **동일한 편집 양식(Template)을 공유** 한다는 중요한 특징이 있습니다. 따라서 문서 내부에 이미 존재하는 **전자 북마크(목차 트리)** 와 **반복되는 소제목 텍스트 패턴** 을 규칙으로 정의하면, 같은 양식의 문서 수백\~수천 페이지를 일관된 표준 Markdown(`H1~H4`)으로 정확하게 정규화할 수 있습니다.

이번 실습에서는 대표적인 예시로 ``AWS Well-Architected Framework 4대 영역(Pillar) 백서 시리즈(총 4권, 793페이지)`` 를 정규화해 보겠습니다. 이 백서 시리즈의 내부 구조와 표준 규격 매핑 방식은 다음과 같습니다.

| PDF 문서 내부의 구조 단서 | 변환할 표준 규격 위치 | 역할 |
|:---|:---|:---|
| 문서 표지 제목 (`# ... - AWS Well-Architected Framework`) | **H1 (`#`)** | 문서 최상위 제목 |
| PDF 전자 북마크 1단계(L1)의 대분류 영역 (`Selection`, `Compute` 등) | **H2 (`##`)** | 대분류 장(Chapter) 제목 (`toc["search"] = 2` 대상) |
| 모범 사례 코드 패턴 (`**PERF02-BP01 ...**`) 및 `**Design principles**` | **H3 (`###`)** | 핵심 모범 사례 제목 (`toc["read"] = 3` 대상) |
| 모범 사례 내부의 고정 소제목 (`**Desired outcome**`, `**Common anti-patterns**`, `**Implementation guidance**`) | **H4 (`####`)** | 세부 항목 헤더 (청크 분할 기준 및 말단 경로) |
| 매 페이지 상단 머리말 및 목차(TOC) 점선 페이지 | **제거** | 불필요한 노이즈 텍스트 삭제 |

범용 변환 함수인 `pdf_to_markdown()`은 그대로 유지하고, 문서 양식에 종속적인 규칙만 설정 객체(`AWS_PDF_OPTIONS`)로 분리해 전달합니다. 이렇게 설계하면 나중에 다른 양식의 PDF 시리즈가 추가되더라도 **변환 코드는 수정하지 않고 설정 프리셋만 새로 추가**하면 됩니다.

#### 테스트

먼저 샘플 PDF 파일에 어떤 전자 북마크(목차) 정보가 들어 있는지 확인해 보겠습니다.

In [ ]:
import pymupdf

aws_pdfs = sorted(glob.glob("data/tech_pdfs/aws_waf_series/*.pdf"))
print(f"📂 발견된 AWS Well-Architected 백서 ({len(aws_pdfs)}개):")
for p in aws_pdfs:
    print(" -", os.path.basename(p))

# 실습에서는 성능 효율성(Performance Efficiency, 124페이지) 백서를 대표 샘플로 살펴봅니다
aws_pdf = [p for p in aws_pdfs if "Performance_Efficiency" in p][0]
doc = pymupdf.open(aws_pdf)
print(f"\n📄 샘플 문서: {os.path.basename(aws_pdf)} ({len(doc)}페이지)\n")
print("=== PDF 전자 북마크 상위 15개 (level, page, title) ===")
for level, title, page in doc.get_toc()[:15]:
    print(f"  L{level} p{page:>3}: {title}")

출력된 북마크를 보면 대분류 아키텍처 영역(`Introduction`, `Architecture selection` 등)은 **레벨 1(L1)**에, 개별 모범 사례(`PERF01-BP01` 등)와 세부 항목(`Implementation guidance`, `Implementation steps`)은 **레벨 2\~4(L2\~L4)**에 체계적으로 정리되어 있습니다.

그렇다면 별도의 정규화 처리 없이 **일반적인 자동화 파서(`pymupdf4llm` 기본 설정)**로 이 PDF를 바로 변환하면 어떻게 될까요? 모범 사례 `PERF02-BP01`(워크로드에 적합한 컴퓨팅 옵션 선택)이 시작되는 27페이지를 기본 파서로 변환해 보겠습니다.

In [ ]:
import pymupdf4llm

bp_page = next(p for lvl, t, p in doc.get_toc() if "PERF02-BP01" in t)
raw_md = pymupdf4llm.to_markdown(doc, pages=[bp_page - 1])

print(f"=== 일반 변환 결과 (p{bp_page}) ===")
print(raw_md[:600])
print("\n📌 헤더로 인식된 줄:", [ln for ln in raw_md.splitlines() if ln.startswith("#")])

일반 파서의 변환 결과를 보면 왜 별도의 정규화 과정이 필요한지 확인할 수 있습니다.

1. **불완전한 헤더 추출**: 파서가 글자 크기에만 의존해 제목을 추정하다 보니, 줄바꿈된 제목(`### **PERF02-BP01 Select the best compute options for your** **workload**`)이 어색하게 분리되었고 불필요한 볼드 기호(`**`)가 섞여 있습니다.
2. **하위 계층 구조 누락**: 해당 페이지 본문 아래에는 `Desired outcome`, `Common anti-patterns`, `Implementation steps`와 같은 핵심 소제목들이 포함되어 있지만, 본문과 글자 크기가 비슷하다는 이유로 마크다운 헤더(`####`)로 변환되지 못하고 일반 볼드체 텍스트(`**...**`)로 남았습니다.
3. **머리말 노이즈 혼입**: 페이지 첫 줄에 `Performance Efficiency Pillar AWS Well-Architected Framework`라는 상단 머리말이 본문처럼 들어와 있습니다.

자동화 파서는 이처럼 내부 계층 구조를 제대로 분석하지 못하더라도 별도의 에러나 경고를 발생시키지 않습니다. 따라서 변환 결과를 그대로 청킹하면 하위 구조가 무너진 상태로 인덱싱됩니다.

이제 동일한 PDF 문서를 정규화 함수(`pdf_to_markdown`)와 백서 전용 규칙(`AWS_PDF_OPTIONS`)을 사용해 표준 규격으로 변환해 보겠습니다.

In [ ]:
from rag.vector.preprocessing import pdf_to_markdown, measure_text_coverage
from rag.vector.preprocessing.presets import AWS_PDF_OPTIONS

aws_md, aws_meta = pdf_to_markdown(aws_pdf, **AWS_PDF_OPTIONS)
cov = measure_text_coverage(aws_pdf, aws_md)

# 정규화 후 생성된 마크다운 4단계 계층 구조 확인 (H1: 백서명, H2: 영역, H3: Best Practice, H4: 세부 항목)
aws_headers = [ln for ln in aws_md.splitlines() if ln.startswith(("# ", "## ", "### ", "#### "))]
print(f"📌 {os.path.basename(aws_pdf)} 정규화 완료 (텍스트 보존율: {cov['ratio']:.2%}, {cov['kept']}/{cov['total']}줄):")
for h in aws_headers[:16]:
    print(" ", h)
print(f"  ... (총 {len(aws_headers)}개 표준 H1~H4 헤더로 정규화 완료)")

In [ ]:
# Part 3에서 동일한 ContextualHeader로 테스트하기 위해 정규화된 Markdown을 저장합니다
os.makedirs("artifacts/normalized", exist_ok=True)
aws_norm_path = f"artifacts/normalized/{os.path.splitext(os.path.basename(aws_pdf))[0]}.md"
with open(aws_norm_path, "w", encoding="utf-8") as f:
    f.write(aws_md)
print("\n💾 정규화된 파일 저장 완료:", aws_norm_path)

💡 정규화 결과, 의도한 규격대로 **H1 = 백서 제목**, **H2 = 대분류 아키텍처 영역**, **H3 = 모범 사례(Best Practice)**, **H4 = 세부 항목(`Desired outcome`, `Common anti-patterns`, `Implementation steps`)**이 일관되게 복원되었습니다.
이제 한글 사내 규정(`.md`)과 영문 기술 백서(`.pdf`) 모두 **완전히 동일한 표준 마크다운 계층 구조(`H1~H4`)**를 갖추게 되었습니다.

### 🔍 정규화 품질 검수: 정량 지표로 변환 결과 확인하기

수백 페이지 분량의 대량 문서를 변환할 때는 모든 페이지를 눈으로 직접 대조하기 어렵습니다. 따라서 실무에서는 정규화가 정상적으로 수행되었는지 다음 두 가지 지표로 검수합니다.

| 검수 지표 | 확인 내용 | 검증 방법 |
|:---|:---|:---|
| **텍스트 보존율** | PDF 원문의 텍스트 줄이 정규화 과정에서 누락되지 않고 보존되었는가 | `measure_text_coverage()`로 원문 대비 보존 비율 계산 |
| **계층 헤더 분포** | 목표한 `H1~H4` 헤더가 모든 문서에서 누락 없이 일관되게 생성되었는가 | 변환된 마크다운의 `H1~H4` 헤더 개수 집계 |

`AWS_PDF_OPTIONS` 설정 하나를 4대 영역 백서 전체(총 793페이지)에 동일하게 적용한 검수 결과는 다음과 같이 모두 **99.7% 이상의 텍스트 보존율**을 기록합니다.

| 문서 파일명 | 페이지 수 | 텍스트 보존율 | 생성된 표준 헤더 분포 |
|:---|---:|---:|:---|
| `Security_Pillar.pdf` | 240p | **99.73%** | `H1=1, H2=8, H3=69, H4=273` |
| `Reliability_Pillar.pdf` | 274p | **99.92%** | `H1=1, H2=6, H3=49, H4=293` |
| `Performance_Efficiency_Pillar.pdf` | 124p | **99.82%** | `H1=1, H2=7, H3=38, H4=128` |
| `Cost_Optimization_Pillar.pdf` | 155p | **99.98%** | `H1=1, H2=7, H3=45, H4=99` |

중요한 점은 어떤 파싱 도구를 사용하든 **최종 출력만 표준 Markdown 규격(`H1~H4`)으로 맞추면, 이후의 청킹·인덱싱·검색 파이프라인은 그대로 재사용할 수 있다**는 것입니다.

---

### 🤖 실무 팁: AI 코딩 에이전트를 활용한 대량 문서 정규화 설정 작성

새로운 문서 양식이 들어올 때마다 사람이 직접 정규식과 북마크 규칙을 짜는 대신, **사람은 목표 규격과 합격 기준만 정의**하고 **규칙 작성과 반복 검수는 AI 코딩 에이전트에게 위임**하면 공수를 크게 줄일 수 있습니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    ENG["1. 엔지니어 정의<br/>• 목표 규격: H1~H4 역할<br/>• 합격선: 텍스트 보존율 ≥ 99%"]

    subgraph LOOP["2. AI 코딩 에이전트 (자율 진단 · 구현 · 검수 루프)"]
        direction LR
        S1["샘플 PDF 진단<br/>북마크 · 소제목 패턴<br/>머리말 노이즈 파악"] --> S2["규칙 프리셋 작성<br/>(예: AWS_PDF_OPTIONS)"]
        S2 --> S3["전 파일 자동 검수<br/>보존율 · H1~H4 분포 측정"]
        S3 -->|"미달 시 규칙 보완"| S2
    end

    PIPE["3. 공용 인덱싱 파이프라인<br/>동일 양식 수백 페이지 일괄 변환<br/>➔ ContextualHeader 청킹 직결"]

    ENG --> S1
    S3 ==>|"검수 통과"| PIPE

    style ENG fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style LOOP fill:#fff8e1,stroke:#f9a825,stroke-width:1.5px
    style PIPE fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

- **엔지니어(사람)**는 우리 시스템의 목표 규격(무엇을 `H1~H4`로 삼고 어떤 메타데이터를 추출할지)과 합격 기준(예: 텍스트 보존율 99% 이상)만 정의합니다.
- **AI 코딩 에이전트**는 샘플 PDF의 북마크와 텍스트 패턴을 진단하여 정규화 설정을 작성하고, 전체 파일이 검수 기준을 통과할 때까지 설정을 반복 개선합니다.
- 한 번 만들어진 설정은 동일한 양식으로 발행되는 후속 문서들에 계속 재사용할 수 있습니다.

<details>
<summary>📋 참고 프롬프트: 새로운 양식의 PDF 정규화 설정 만들기 (3단계)</summary>

**1단계: 문서 구조 진단**
```text
'{pdf_path}' PDF를 표준 Markdown 규격(H1 = 최상위 문서명, H2 = 대분류 장, H3 = 핵심 조항/주제, H4 = 세부 항목)으로
정규화하려고 해. 이 문서가 가진 구조 정보를 먼저 조사해 줘.
1. 전자 북마크(doc.get_toc())가 있는지, 깊이별 항목과 시작 페이지
2. 반복되는 소제목, 머리말·목차 점선 같은 노이즈의 표기 패턴
3. pymupdf4llm 기본 변환에서 누락되거나 잘못된 헤더로 인식되는 부분이 있는지
```

**2단계: 정규화 설정 작성**
```text
진단 결과를 바탕으로 pdf_to_markdown() 함수에 전달할 옵션 설정 dict를 작성해 줘.
변환 함수 본체는 수정하지 말고, 결정론적 규칙(북마크 매핑, 정규식 패턴)만 사용해 줘.
```

**3단계: 전체 파일 품질 검수**
```text
작성한 설정을 같은 시리즈의 모든 PDF 파일에 적용해서 파일별 검수 표를 만들어 줘.
1. measure_text_coverage()로 계산한 텍스트 보존율
2. ContextualHeader로 청킹했을 때 H1~H4 계층별 청크 수
보존율이 99% 미만이거나 규격에 맞지 않는 헤더가 발생한 파일이 있으면 원인을 분석해 설정을 수정해 줘.
```
</details>

---
## 📝 Part 3. 문맥 보존 청킹: "작게 찾고, 크게 읽는다" (검색용·읽기용 문맥 헤더 분리)

Part 2의 정규화 과정을 통해 모든 문서에 표준 계층(`H1~H4`)이 만들어졌습니다. 이제 문서를 청크로 자르면서 각 청크에 상위 계층 정보를 붙여 **문맥 유실(탈맥락화) 문제**를 해결할 차례입니다.

청킹과 검색을 설계할 때는 **검색할 때 유리한 크기**와 **LLM이 읽고 답변할 때 유리한 크기**가 서로 다르다는 점을 고려해야 합니다. 이를 해결하는 핵심 설계 원리가 바로 **"작게 찾고, 크게 읽는다 (Small-to-Big)"**입니다.

| 청크 크기 | 검색 시점 (임베딩 유사도 및 검색 정밀도) | 열람 시점 (LLM 답변 생성 및 연관 조항 파악) |
|:---|:---|:---|
| **작게 구성할 때** (300\~500자) | ✅ **유리함** — 한 청크에 하나의 주제만 담겨 임베딩 벡터의 주제가 뚜렷함 | ❌ **불리함** — 표나 단서 조항이 앞뒤로 잘리고 문서 내 다른 조항을 파악하기 어려움 |
| **크게 구성할 때** (2,000자 이상) | ❌ **불리함** — 여러 주제와 긴 목차가 한꺼번에 섞여 임베딩 벡터가 흐려짐 | ✅ **유리함** — 주변 맥락과 문서 전체 구조가 온전히 포함되어 정확한 답변 가능 |

이 두 가지 요구사항을 동시에 만족시키기 위해 다음 **두 가지 기법**을 결합합니다.
1. **문맥 헤더의 이원화 (`ContextualHeader` — `toc={"search": 2, "read": 3}`)**:
   - **작게 찾기 (`search: 2`)**: 벡터 임베딩과 BM25 색인에 사용되는 `page_content`에는 **대분류 장 목차(`H1~H2`)와 현재 위치 경로(Breadcrumb)**만 간결하게 넣어 본문 의미가 희석되지 않도록 합니다.
   - **크게 읽기 (`read: 3`)**: 검색된 뒤 에이전트에게 전달할 `metadata["doc_header"]`에는 **세부 조항 목차(`H1~H3`)**까지 포함시켜, 에이전트가 문서 전체 구성을 파악하고 연관된 다른 조항을 **추가 검색(Re-retrieval)**할 수 있도록 돕습니다.
2. **인접 청크 윈도우 확장 (`ContextualWindow` — Part 6에서 다룸)**:
   - **작게 찾기**: 검색 자체는 조항 단위의 작은 청크로 정밀하게 수행합니다.
   - **크게 읽기**: 검색된 청크를 에이전트에게 반환할 때는 앞뒤로 이웃한 청크의 본문(`metadata["raw_content"]`)을 함께 이어 붙여 잘린 표와 단서 조항을 복원합니다.

### 왜 목차를 검색용(`search: 2`)과 읽기용(`read: 3`) 두 벌로 분리할까요?

한 문서에 조항(`H3`)이 수십 개 있을 때 모든 조항 제목(약 1,400자)을 350자짜리 작은 청크마다 그대로 넣어 임베딩하면, 정작 본문보다 공통 목차의 비중이 훨씬 커져 **같은 문서의 모든 청크 벡터가 서로 비슷해지는 임베딩 희석 현상**이 발생합니다. 반대로 목차를 아예 빼 버리면, 에이전트가 검색된 조항을 읽더라도 같은 문서 안에 어떤 연관 조항(예: 정산 기한 `제20조`, 부당 수령 제재 `제26조`)이 더 있는지 알 수 없습니다.

따라서 표준 Markdown 계층에서 **검색 색인용 요약 목차(`H1~H2`)** 와 **에이전트 열람용 전체 목차(`H1~H3`)** 를 분리해 추출합니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    subgraph MD["1. 표준 Markdown 문서 계층 (국내 여비 규정)"]
        H12["[상위 계층: H1 문서명 + H2 장 제목]<br/>총 6개 장 · 약 180자<br/># 국내 여비 규정 / ## 제1장 총칙 ~ ## 제6장 보칙"]
        H3_ALL["[중간 계층: H1 ~ H3 세부 조항 제목]<br/>총 32개 조항 · 약 1,400자<br/>### 제1조 목적 ... ### 제15조 일비·식비 ... ### 제28조"]
        BODY["[하위 본문: 현재 분할된 청크 (약 350자)]<br/>소속: 제3장 > 제15조<br/>'2. 당일 출장은 식비 12,000원, 일비 50% 지급...'"]
    end

    subgraph OUT["2. 목적에 맞춰 분리 추출한 3가지 문맥 헤더"]
        TOC_S["① 검색용 요약 목차 (toc.search = 2)<br/>짧은 장 목차(~180자)만 임베딩에 포함<br/>➔ 본문(350자) 임베딩 벡터 희석 방지"]
        TOC_R["② 읽기용 전체 목차 (toc.read = 3)<br/>32개 전체 조항 목차(~1,400자) 별도 보관<br/>➔ 검색 후 에이전트의 연관 조항 탐색 유도"]
        BC["③ 현재 위치 경로 (Breadcrumb)<br/>'국내 여비 지급 규정 > 제3장 > 제15조'<br/>➔ 잘려 나온 하위 청크의 소속 조항 복원"]
    end

    H12 ==>|"H1~H2만 추출"| TOC_S
    H3_ALL ==>|"H1~H3 전체 추출"| TOC_R
    BODY ==>|"상위 경로 추적"| BC

    style BODY fill:#fff3e0,stroke:#f57c00,stroke-width:1.5px
    style TOC_S fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style TOC_R fill:#fff8e1,stroke:#f9a825,stroke-width:1.5px
    style BC fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

`ContextualHeader`는 문서를 두 단계(`1차: H1~H4 헤더 기준 논리 분할` ➔ `2차: chunk_size 기준 길이 분할`)로 나눈 뒤, 하나의 `Document` 객체 안에 **검색 색인에 쓰일 `page_content`** 와 **검색 후 열람에 쓰일 `metadata`** 를 이원화하여 저장합니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    subgraph DOC_OBJ["1. 단일 청크 Document 객체 (이원화 저장)"]
        direction TB
        subgraph PC["Document.page_content (검색 색인용 · 약 600자)"]
            direction TB
            PC_H["[짧은 검색용 헤더]<br/>문서명·소관부서 + 요약 목차(H1~H2) + 현재 경로"]
            PC_B["[청크 본문]<br/>당일 출장 시 식비 12,000원, 일비는 기준액의 50%..."]
            PC_H --> PC_B
        end

        subgraph META["Document.metadata (검색 후 열람·복원용)"]
            direction TB
            M_H["metadata['doc_header']<br/>문서명·현재 경로 + 전체 조항 목차(H1~H3)"]
            M_R["metadata['raw_content']<br/>헤더를 제외한 순수 본문 (인접 청크 결합용)"]
        end
    end

    subgraph CONSUME["2. 시점별 활용 방식"]
        direction TB
        IDX["① 인덱싱 · 검색 시점<br/>Vector 임베딩 & BM25 색인 입력<br/>(짧은 헤더 덕분에 본문 주제 선명도 유지)"]
        READ_OUT["② 검색 후 에이전트 열람 시점 (Part 6 연동)<br/>상단: doc_header(H1~H3 전체 목차) 1회 부착<br/>본문: 앞뒤 이웃 청크의 raw_content 순서 결합"]
    end

    PC ==>|"색인 텍스트"| IDX
    M_H ==>|"상단 헤더 교체"| READ_OUT
    M_R ==>|"이웃 본문 병합"| READ_OUT

    style PC fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style META fill:#fff8e1,stroke:#f9a825,stroke-width:1.5px
    style IDX fill:#ede7f6,stroke:#5e35b1,stroke-width:1.5px
    style READ_OUT fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
```

| 구성 요소 | 저장 위치 | 핵심 역할 및 기대 효과 |
|:---|:---|:---|
| **현재 위치 경로 (Breadcrumb)** | `page_content` 및 `metadata["breadcrumb"]` | 해당 청크가 어느 문서의 몇 장 몇 조인지 명시하여 질문과의 검색 매칭 정확도를 높임 |
| **검색용 요약 목차 (`search: 2`)** | `page_content` 상단 헤더 | 대분류 장(`H1~H2`) 목차만 짧게 포함하여 긴 목차로 인해 본문 임베딩 벡터가 희석되는 현상을 방지 |
| **읽기용 전체 목차 (`read: 3`)** | `metadata["doc_header"]` | 검색 후 에이전트가 결과를 읽을 때 전체 조항(`H1~H3`) 목록으로 교체 제공하여 연관 조항 재검색을 유도 |
| **순수 본문 (`raw_content`)** | `metadata["raw_content"]` | Part 6에서 앞뒤 이웃 청크를 이어 붙일 때 헤더가 중복 삽입되지 않도록 순수 본문만 별도 보관 |

이제 마크다운 문서(사내 규정)와 정규화된 PDF 문서(기술 백서)에 동일한 `ContextualHeader`를 적용하여 결과를 비교해 보겠습니다.

### 3.1 마크다운 문서의 문맥 헤더 구성 확인 (예시: 사내 규정)

In [ ]:
from rag.vector.indexing import ContextualHeader

# 기본값: toc={"search": 2, "read": 3} (검색용은 H1~H2 장 목차, 읽기용은 H1~H3 전체 조항 목차)
# Part 6의 윈도우 확장(인접 청크 결합) 시 경계 문장이 중복되지 않도록 chunk_overlap=0으로 설정합니다.
header_chunker = ContextualHeader(chunk_size=500, chunk_overlap=0, toc={"search": 2, "read": 3})

In [ ]:
# 1. 사내 규정 문서 청킹 및 문맥 헤더 주입
policy_chunks, _ = header_chunker.chunk_file(policy_files[0], doc_id=0)

sample_pol = next(c for c in policy_chunks if "제15조" in c.metadata.get("breadcrumb", ""))

print(f"📊 글자 수 비교: 순수 본문({len(sample_pol.metadata['raw_content'])}자) | 검색용 page_content({len(sample_pol.page_content)}자) | 읽기용 doc_header({len(sample_pol.metadata['doc_header'])}자)\n")
print("=== [1] 검색용 청크 (page_content: H1~H2 장 목차 + 본문 → 임베딩 대상) ===")
print(sample_pol.page_content)

In [ ]:
print("\n=== [2] 읽기용 헤더 (metadata['doc_header']: H1~H3 전체 조항 목차 → 에이전트 열람용) ===")
print(sample_pol.metadata["doc_header"][:900] + "\n... (하략)")

### 3.2 정규화된 PDF 문서에 동일한 청킹 파이프라인 적용 (예시: 기술 백서)

Part 2에서 표준 Markdown(`H1~H4`)으로 정규화해 두었던 기술 백서 파일(`aws_norm_path`)에도 방금 위에서 생성한 **`header_chunker` 인스턴스(`chunk_size=500, chunk_overlap=0`)**를 그대로 적용합니다.

In [ ]:
# 2. AWS 백서(정규화된 Markdown)에도 동일한 header_chunker(chunk_size=500, chunk_overlap=0)를 그대로 적용
aws_chunks, _ = header_chunker.chunk_file(aws_norm_path, doc_id=1)

sample_aws = next(
    c for c in aws_chunks
    if "PERF02-BP01" in c.metadata.get("breadcrumb", "") and "Common anti-patterns" in c.metadata.get("breadcrumb", "")
)

print(f"📊 글자 수 비교: 순수 본문({len(sample_aws.metadata['raw_content'])}자) | 검색용 page_content({len(sample_aws.page_content)}자) | 읽기용 doc_header({len(sample_aws.metadata['doc_header'])}자)\n")
print("=== [1] 원본 청크 본문 (raw_content - 상위 맥락이 없으면 어떤 영역의 지침인지 알기 어려움) ===")
print(sample_aws.metadata["raw_content"][:300])

In [ ]:
print("\n=== [2] 검색용 청크 (page_content: H1~H2 장 목차 + 현재 위치 경로 + 본문) ===")
print(sample_aws.page_content)

In [ ]:
print("\n=== [2] 읽기용 헤더 (metadata['doc_header']: H1~H3 전체 조항 목차 → 에이전트 열람용) ===")
print(sample_aws.metadata["doc_header"][:900] + "\n... (하략)")

💡 **실습 결과 정리:**

1. **출처와 위치 맥락의 보존**: 본문만 잘려 나갔을 때는 어느 문서의 어떤 조항인지 알 수 없었던 청크 상단에 `[문서명]`과 `[현재 위치 경로: ... > Compute > PERF02-BP01 ... > Common anti-patterns]`가 자동으로 추가되어, 청크 스스로 자신의 상위 맥락을 설명할 수 있게 되었습니다.
2. **단일 청킹 파이프라인 재사용**: 사내 규정(`.md`)과 기술 백서(`.pdf`)를 처리할 때 **`ContextualHeader`의 코드는 단 한 줄도 수정하지 않았습니다.** 전처리 단계에서 표준 Markdown(`H1~H4`)으로 규격을 통일했기 때문에 이후의 청킹 로직을 그대로 공유할 수 있습니다.

> 💡 **왜 목차(TOC)를 검색용(`search: 2`)과 읽기용(`read: 3`)으로 분리할까요?**  
> 에이전트가 첫 번째 검색 결과를 읽은 뒤 연관된 다른 조항(예: 여비 규정 `제15조`를 확인한 뒤 징계 기준인 `제45조`를 추가로 검색)을 찾아가려면 문서 전체의 조항 목록(`H1~H3`)을 볼 수 있어야 합니다.  
> 하지만 수십 개의 조항 제목이 나열된 긴 목차(1,200\~2,500자)를 모든 청크의 `page_content`에 그대로 넣어 임베딩하면, 정작 짧은 본문(300\~500자)보다 공통 목차의 비중이 커져 **같은 문서의 모든 청크 임베딩 벡터가 서로 비슷해지는 문제(임베딩 희석)**가 발생합니다.  
> 따라서 검색 대상인 `page_content`에는 짧은 대분류 장 목차(`H1~H2`, 약 150\~200자)만 넣어 검색 변별력을 유지하고, 검색이 완료된 후 에이전트에게 컨텍스트를 반환할 때만 `metadata["doc_header"]`의 전체 조항 목차(`H1~H3`)로 교체해 전달합니다.

---
## 🔗 Part 4. 단일 파이프라인 통합 인덱싱과 도메인 메타데이터 태깅

지금까지 살펴본 정규화(`to_markdown`)와 문맥 헤더 청킹(`ContextualHeader`) 과정을 하나의 진입점 함수인 `index_documents()`로 묶어 실행해 보겠습니다. 이 함수는 입력된 파일들의 확장자를 확인하여 적절한 정규화기를 거친 뒤, 동일한 청킹 파이프라인으로 일괄 처리합니다.

### 도메인 메타데이터 태깅: 문맥 헤더(Soft)와 메타데이터 필터(Hard)의 결합

동일한 양식의 문서 시리즈를 대량으로 인덱싱할 때는 **문서 카테고리나 도메인 영역을 메타데이터로 함께 태깅**해 두는 것이 효과적입니다.
예를 들어 AWS Well-Architected 4대 영역 백서(`Security`, `Reliability`, `Performance Efficiency`, `Cost Optimization`)는 모든 문서에 `### Design principles`, `### Definition`처럼 동일한 제목의 섹션이 들어 있으며, 컴퓨팅·스토리지·모니터링 같은 주제를 각자의 관점(보안 vs 신뢰성 vs 성능 vs 비용)에서 다룹니다.

이때 각 청크에 문서 영역을 나타내는 메타데이터(`pillar: "security" | "reliability" | "performance" | "cost"`)를 함께 부여하면, **문맥 헤더를 통한 의미적 매칭(Soft)**과 **메타데이터 조건을 통한 엄격한 필터링(Hard)**을 상황에 맞게 조합할 수 있습니다.

| 구분 | 문맥 헤더 주입 (Soft Matching) | 메타데이터 필터링 (Hard Filtering) |
|:---|:---|:---|
| **저장 위치** | 임베딩되는 본문 텍스트 상단 (`[문서명: Performance Efficiency...]`) | 벡터 DB의 메타데이터 필드 (`pillar: "performance"`) |
| **동작 방식** | 질문과 문서 영역 간의 의미 유사도를 높여 관련 청크의 검색 순위를 끌어올림 | 지정된 조건과 일치하지 않는 다른 영역의 문서 청크를 **검색 대상에서 완전히 제외** |
| **장점** | 사용자가 특정 영역을 명시하지 않은 포괄적 질문에서도 문맥을 반영해 검색 가능 | 특정 영역(예: 보안)만 찾고 싶을 때 다른 백서의 동명 섹션이 섞여 들어올 가능성을 원천 차단 |
| **주의점** | 본문 표현이 매우 유사하면 다른 영역의 백서 청크가 일부 함께 검색될 수 있음 | 여러 영역을 비교하는 질문(예: "성능 효율성과 비용 최적화의 트레이드오프")에 단일 필터를 걸면 필요한 문서가 제외됨 |

> 💡 **실무 팁:** 실무에서는 둘 중 하나만 선택하는 대신 **두 가지를 모두 적용**합니다. 인덱싱 시점에 메타데이터를 함께 태깅해 두고, 에이전트가 호출하는 검색 도구(예: `search_aws_guides`)에 선택적 필터 인자(`pillar`)를 열어 두면 **에이전트가 질문의 성격을 보고 필터를 걸지, 전체 영역에서 검색할지를 스스로 판단**할 수 있습니다.

In [ ]:
from collections import Counter
from rag.vector.indexing import index_documents
from rag.vector.preprocessing.presets import aws_metadata

mixed_paths = policy_files + [aws_pdf]
chunks, lookup = index_documents(
    mixed_paths,
    header_chunker,
    pdf_options=AWS_PDF_OPTIONS,
    extra_metadata=lambda path, meta: aws_metadata(path, meta) if path.endswith(".pdf") else {},
)

print(f"✅ 총 {len(chunks)}개 청크 통합 인덱싱 완료")
for src, n in Counter(c.metadata["source_file"] for c in chunks).items():
    print(f"  • {src}: {n}개")

aws_chunk = next(c for c in chunks if c.metadata.get("pillar") and "PERF02-BP01" in c.metadata["breadcrumb"])
print("\n=== PDF에서 나온 청크도 사내 규정과 동일한 형식의 헤더를 가집니다 ===")
print(aws_chunk.page_content.split("-------------------------------")[0])
print("=== 청크 메타데이터 (아키텍처 영역 태깅) ===")
print({k: aws_chunk.metadata[k] for k in ("pillar", "pillar_name", "framework", "breadcrumb")})

---
## 🔎 Part 5. 하이브리드 검색: 의미 검색(Dense)과 키워드 검색(BM25)의 결합

### 왜 순수 벡터 검색(Dense Search)만으로는 부족할까요?

임베딩 모델을 이용한 **밀집 벡터 검색(Dense Search)**은 문장의 전반적인 의미를 파악하는 데 강점이 있습니다. 예를 들어 사용자가 *"출장 가면 밥값은 얼마 줘?"*라고 질문했을 때, 문서에 `밥값` 대신 `식비`라는 단어가 적혀 있어도 의미적 유사성을 통해 관련 조항을 찾아낼 수 있습니다.

반면, 현업 문서 검색에서 자주 등장하는 **정확한 조항 번호(`제15조`), 영문 약어(`DLP`), 코드명(`PERF02-BP01`)**처럼 글자 하나라도 달라지면 안 되는 질의에서는 단어 일치 기반의 **키워드 검색(BM25)**이 훨씬 정확하게 동작합니다. 또한 사내에서 흔히 쓰는 줄임말(예: `클운팀`)은 검색 전에 정식 명칭(`클라우드운영팀`)으로 변환해 주어야 두 검색기 모두 제 성능을 발휘할 수 있습니다.

| 질문 유형 | Dense (의미 기반) | BM25 (키워드 기반) | 특징 및 보완 관계 |
|:---|:-:|:-:|:---|
| "출장 가면 밥값은 얼마 줘?" | ✅ 강함 | ❌ 취약 | 문서에는 `식비`로 표기됨 — 단어는 달라도 의미가 같은 경우 Dense 검색이 유리 |
| "제15조 내용 알려줘" | △ 보통 | ✅ 강함 | 조항 번호는 의미 추론이 아니라 **정확한 문자열 일치**가 중요하므로 BM25가 유리 |
| "DLP 승인 절차" | △ 보통 | ✅ 강함 | 전문 약어나 고유명사는 임베딩 공간에서 변별력이 떨어지기 쉬움 |
| "클운팀 출장 결재" | △ 보통 | △ 보통 | 문서에는 정식 명칭(`클라우드운영팀`)만 존재 — **용어 사전**으로 정규화 선행 필요 |

### 하이브리드 검색 파이프라인 구조

이러한 상호 보완적 특성을 살리기 위해 **① 사내 용어 사전 정규화 ➔ ② BM25 및 Dense 병렬 검색 ➔ ③ RRF 순위 융합 ➔ ④ 윈도우 확장** 순서로 검색 파이프라인을 구성합니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    Q["사용자 질문<br/>'클운팀 제15조 밥값 기준'"] --> D["1. 용어 사전 치환<br/>'클운팀' → '클라우드운영팀'"]
    D --> S["2A. BM25 키워드 검색<br/>'제15조', '클라우드운영팀'<br/>문자열 정확 일치 순위"]
    D --> V["2B. Dense 벡터 검색<br/>'밥값' ≈ '식비 지급 기준'<br/>의미 유사도 순위"]
    S --> F["3. RRF 순위 융합<br/>점수 단위 차이를 제거하고<br/>양쪽 상위권 청크 통합 선정"]
    V --> F
    F --> W["4. 윈도우 확장 (Part 6)<br/>선정된 중심 청크 앞뒤의<br/>기준표·단서 조항 결합 반환"]

    style D fill:#fff8e1,stroke:#f9a825,stroke-width:1.5px
    style S fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style V fill:#ede7f6,stroke:#5e35b1,stroke-width:1.5px
    style F fill:#fff3e0,stroke:#f57c00,stroke-width:1.5px
    style W fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

| 단계 | 수행 작업 | 핵심 동작 원리 |
|:---:|:---|:---|
| **1단계** | 도메인 용어 사전 정규화 (`DomainDictionary.normalize`) | 정규식 사전을 이용해 질문 속 사내 약어·은어를 공식 명칭으로 치환 (별도 LLM 호출 없음) |
| **2단계** | BM25 및 Dense 병렬 검색 (`HybridRetriever.search`) | 키워드 일치 기반의 BM25 검색과 임베딩 유사도 기반의 Dense 검색을 각각 수행 |
| **3단계** | RRF 결과 통합 (`reciprocal_rank_fusion`) | 서로 다른 두 검색기의 결과를 순위 기준으로 결합하여 최종 상위 문서 선정 |

### RRF (Reciprocal Rank Fusion): 왜 점수가 아니라 '순위'로 합칠까요?

Dense 검색이 반환하는 코사인 유사도 점수(보통 `0 ~ 1` 사이)와 BM25 검색이 반환하는 키워드 매칭 점수(문서 길이와 단어 빈도에 따라 `0 ~ 수십`까지 나오는 비정규화 값)는 **점수의 단위와 분포가 완전히 다르기 때문에 단순 합산할 수 없습니다.**

이를 해결하기 위해 **RRF(Reciprocal Rank Fusion)** 알고리즘은 각 검색기가 매긴 절대 점수 대신 **몇 등을 했는지(순위, Rank)의 역수**만을 더해 최종 점수를 계산합니다.

$$\text{RRF}(d) = \sum_{r \in \{\text{BM25},\ \text{Dense}\}} \frac{1}{k + \text{rank}_r(d)} \qquad (k = 60)$$

| 검색된 청크 | BM25 순위 | Dense 순위 | RRF 점수 계산 ($k=60$) | 최종 순위 |
|:---|:-:|:-:|:---|:-:|
| 청크 A | 1위 | 2위 | $1/(60+1) + 1/(60+2) =$ **0.0325** | **1위** |
| 청크 C | 3위 | 3위 | $1/(60+3) + 1/(60+3) =$ **0.0317** | **2위** |
| 청크 B | 순위권 밖 | 1위 | $1/(60+1) =$ **0.0164** | **3위** |

표에서 볼 수 있듯, 한쪽 검색기에서만 1위를 한 문서(청크 B)보다 **의미 검색과 키워드 검색 양쪽에서 모두 상위권에 오른 문서(청크 A, C)**가 더 높은 최종 순위를 얻게 됩니다.

> 💡 **한국어 환경에서 BM25 사용 시 참고사항**:  
> 기본 `BM25Retriever`는 공백(띄어쓰기)을 기준으로 단어를 나눕니다. 한국어는 단어 뒤에 조사가 붙기 때문에(`출장비는`, `출장비를`), 상용 서비스 환경에서는 `preprocess_func` 인자에 한국어 형태소 분석기(예: `Kiwi`)를 연결해 명사 단위로 토큰화하면 키워드 매칭 정확도를 더욱 높일 수 있습니다.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from rag.vector.retrieval import HybridRetriever

# 노트북 실습용 인메모리 인덱스 (app/database의 운영 DB는 건드리지 않습니다)
dense_store = Chroma.from_documents(
    chunks, OpenAIEmbeddings(model="text-embedding-3-large"), collection_name="nb2_demo"
)
bm25 = BM25Retriever.from_documents(chunks, k=5)

# (참고: 이미 인덱싱된 Chroma 컬렉션에서는 HybridRetriever.from_chroma(dense_store) 한 줄로도 생성 가능)
hybrid = HybridRetriever(
    vectorstore=dense_store,
    bm25_retriever=bm25,
    chunk_lookup_map=lookup,
    use_hyde=False,          # HyDE는 LLM 호출이 필요하므로 실습에서는 끔
    use_window_expansion=True,
    window_size=2,
)
print("✅ 하이브리드 검색기 준비 완료")
print("약어 사전 치환 예:", hybrid.domain_dict.normalize("클운팀 직원 출장 결재는 누가 최종 승인해?"))

In [ ]:
query = "클운팀 직원 출장 결재는 누가 최종 승인해?"

print("🔹 Dense 단독 검색 Top-2")
for d in dense_store.similarity_search(query, k=2):
    print("   ", d.metadata["breadcrumb"])

print("\n🔸 하이브리드 검색 Top-2 (약어 사전 + BM25 + RRF + 윈도우 확장)")
for d in hybrid.search(query, top_k=2):
    print("   ", d.metadata["breadcrumb"], f"(청크 {d.metadata.get('window_start_chunk')}~{d.metadata.get('window_end_chunk')} 결합)")

💡 위 실행 결과에서 눈여겨볼 점은 하이브리드 검색기가 반환한 결과 옆의 `(청크 19~20 결합)` 표시입니다.
검색할 때는 작은 청크 단위로 정확히 찾았지만, 결과를 반환할 때는 **찾아낸 중심 청크의 앞뒤 이웃 청크까지 하나로 묶어서** 돌려주었습니다. 이 메커니즘을 다음 파트에서 자세히 살펴보겠습니다.

---
## 🪟 Part 6. 작게 찾고 크게 읽기 ②: 인접 청크 윈도우 확장 (`ContextualWindow`)

문서를 500자 크기의 작은 청크로 나누면 검색 정확도는 높아지지만, **하나의 기준표가 앞 청크에 남거나 본문 바로 뒤에 이어지는 예외·단서 조항이 다음 청크로 넘어가는 현상(수평적 단절)** 이 자주 발생합니다. 검색된 단일 청크만 에이전트에게 전달하면 잘려 나간 기준표나 예외 조건을 보지 못해 오답을 낼 수 있습니다.

**윈도우 확장(`ContextualWindow`)**은 검색 단계에서는 작은 청크로 정밀하게 매칭한 뒤, 결과를 반환하기 직전에 **같은 문서 내의 앞뒤 이웃 청크(`-window_size ~ +window_size`)의 순수 본문(`raw_content`)을 하나로 이어 붙여 완결된 조항으로 복원**하는 기법입니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart TB
    subgraph SEQ["1. 인덱스에 연속 저장된 작은 청크들 (chunk_overlap = 0, 확장 대상: 청크 13 ~ 17)"]
        direction LR
        C13["청크 13 (앞 2)<br/>제14조 숙박비<br/>예외 승인"] --- C14["청크 14 (앞 1)<br/>제15조 1항<br/>정액 기준표"] --- C15["🎯 청크 15 (검색 적중)<br/>제15조 2항<br/>당일 출장 일비 50%"] --- C16["청크 16 (뒤 1)<br/>제15조 3~4항<br/>식사 제공 1/3 공제"] --- C17["청크 17 (뒤 2)<br/>제16조 해외 출장<br/>적용 범위"]
    end

    WIN["2. ContextualWindow 확장 조회 (window_size = 2)<br/>(doc_id, chunk_id) 룩업 맵으로 중심 청크(15) 앞뒤 ±2 이웃을 즉시 조회하고 개별 검색용 헤더는 제거"]

    OUT_BOX["3. 에이전트에게 반환되는 통합 컨텍스트<br/>• 상단(1회 부착): 중심 청크(15)의 읽기용 헤더 (doc_header — H1~H3 전 조항 목차)<br/>• 본문(순서 결합): 청크 13~17의 raw_content 연결 ➔ 기준표(14) + 당일 감액(15) + 공제 단서(16) 통합 복원"]

    SEQ ==>|"중심 청크 기준 ±2 범위 추출"| WIN
    WIN ==>|"읽기용 헤더 1회 + 순수 본문 병합"| OUT_BOX

    style C15 fill:#fff3e0,stroke:#f57c00,stroke-width:2.5px
    style C14 fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style C16 fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style WIN fill:#fff8e1,stroke:#f9a825,stroke-width:1.5px
    style OUT_BOX fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
```

| 구성 요소 | 동작 방식 및 설계 이유 |
|:---|:---|
| **청크 룩업 맵 (`{(doc_id, chunk_id): 청크}`)** | 인덱싱 시점에 생성해 둔 딕셔너리를 사용해, 벡터 DB를 재검색하지 않고도 앞뒤 이웃 청크(`chunk_id - window_size` ~ `chunk_id + window_size`)를 즉시 조회합니다. |
| **읽기용 헤더 1회 부착 (`doc_header`)** | 결합된 텍스트 맨 위에는 중심 청크의 `metadata["doc_header"]`(`H1~H3` 전체 조항 목차 포함)를 **단 한 번만** 붙입니다. |
| **순수 본문 연결 (`raw_content` & `chunk_overlap=0`)** | 앞뒤 청크들은 헤더가 제거된 `metadata["raw_content"]`만 가져와 이어 붙입니다. 청킹 단계에서 `chunk_overlap=0`으로 설정했기 때문에 인접 청크를 합쳐도 경계 문장이 중복되지 않습니다. |
| **확장 범위 (`window_size=2`)** | 중심 청크 기준 앞뒤 2개씩(최대 5개 청크) 결합하여, 여러 조각으로 나뉜 표나 조항 전체를 완결된 문맥으로 복원합니다. |

In [ ]:
expanded = hybrid.search("당일 출장 일비 식비 지급 기준", top_k=1)[0]
print(f"📍 {expanded.metadata['breadcrumb']} | 청크 {expanded.metadata['window_start_chunk']}~{expanded.metadata['window_end_chunk']} 결합\n")
print(expanded.page_content)

💡 실행 결과를 보면 검색에 매칭된 중심 청크(당일 출장 감액 기준)뿐만 아니라, 앞뒤 청크에 있던 정액 지급 기준표와 세부 조항까지 하나의 완결된 텍스트로 결합되었습니다. 덕분에 표나 단서 조항이 청크 경계에서 잘리더라도 LLM은 전체 맥락을 온전히 파악하여 답변할 수 있습니다.

---
## 🕸️ Part 7. 지식 그래프의 원리: 도메인 온톨로지 기반 추출과 다중 홉 탐색

### 왜 벡터 검색만으로는 복합 관계(Multi-Hop) 질문에 답하기 어려울까요?

지금까지 구축한 하이브리드 검색기는 특정 조항이나 기술 지침이 담긴 문단을 찾아내는 데는 매우 뛰어납니다. 하지만 다음과 같이 **여러 문서나 문단에 흩어진 사실들을 징검다리처럼 연결해야 답이 나오는 질문**에서는 한계를 보입니다.

> *"클라우드운영팀 김철수 수석이 총괄하는 프로젝트의 예산을 최종 승인하는 본부장은 누구인가?"*

벡터 검색이나 키워드 검색으로 이 질문을 던지면 `김철수 수석`이나 `클라우드운영팀`이 언급된 문단은 찾을 수 있습니다. 하지만 정작 정답인 `박영희 전무`가 적힌 문단에는 `김철수 수석`이라는 단어가 직접 등장하지 않고 `P-01 프로젝트의 예산 승인권자`라고만 적혀 있기 때문에 검색 상위권에 올라오지 못합니다.

이 질문에 정확히 답하려면 아래와 같이 개체(Entity) 간의 관계를 따라가는 **다단계 탐색(Multi-Hop Traversal)**이 필요합니다.

1. **1-Hop (담당 프로젝트 확인)**: `[김철수 수석]` ──(총괄PM)──▶ `[P-01: 클라우드 네이티브 마이그레이션 (예산 3억 5,000만원)]`
2. **1-Hop (소속 부서 확인)**: `[김철수 수석]` ──(팀장)──▶ `[클라우드운영팀]`
3. **2-Hop (상위 조직 확인)**: `[클라우드사업본부]` ──(산하조직)──▶ `[클라우드운영팀]`
4. **2-Hop (예산 승인권자 확인)**: `[박영희 전무]` ──(예산승인권자)──▶ `[P-01: 클라우드 네이티브 마이그레이션]`

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    A["김철수 수석<br/>(Person)"] -->|"팀장"| B["클라우드운영팀<br/>(Department)"]
    A -->|"총괄PM"| C["P-01 프로젝트<br/>(Project)"]
    D["클라우드사업본부<br/>(Department)"] -->|"산하조직"| B
    E["박영희 전무<br/>(최종 승인권자)"] -->|"예산승인"| C
    E -->|"본부장"| D

    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style C fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style E fill:#e8f5e9,stroke:#388e3c,stroke-width:3px
```

### 지식 그래프 구축 시 흔히 겪는 문제: 어휘와 명칭의 파편화

텍스트에서 지식 그래프를 만들 때는 보통 LLM을 이용해 `(주어, 관계, 목적어)` 형태의 **트리플렛(Triplet)**을 추출합니다. 그런데 아무런 제약 없이 자유 문자열로 추출하게 두면 그래프가 제대로 연결되지 않는 두 가지 문제가 발생합니다.
1. **관계 어휘의 불일치**: 동일한 의미의 관계를 `'팀장이다'`, `'이끈다'`, `'총괄한다'`, `'소속됨'`처럼 문장마다 다르게 추출하여 일관된 탐색이 어렵습니다.
2. **동일 개체의 노드 분리 (Entity Fragmentation)**: 같은 대상이라도 문맥에 따라 `'클운팀'`, `'클라우드운영팀'` 또는 `'김철수'`, `'김철수 수석'`, `'김철수 팀장'`으로 다르게 적혀 있으면 그래프상에서 별개의 노드로 쪼개져 **2-Hop 탐색 경로가 중간에서 끊어집니다.**

### 해법: 도메인 온톨로지(Ontology) 스키마 제약과 엔티티 정규화

이 문제를 해결하기 위해 Pydantic의 `Literal` 타입을 활용하여 LLM이 추출할 수 있는 **개체 유형(`EntityType`)**과 **관계 어휘(`RelationType`)**를 사전에 정의된 목록으로 제한하고, 약어나 직함이 섞인 이름을 대표 명칭으로 통일(`canonicalize_entity()`)합니다.

| 구성 요소 | 허용 값 및 동작 | 설계 목적 |
|:---|:---|:---|
| **개체 유형 (`EntityType`)** | `Person`, `Department`, `Project`, `PolicyOrRule`, `RiskOrGoal` | 그래프 노드의 성격을 명확히 분류하고 시각화 색상을 구분 |
| **관계 어휘 (`RelationType`)** | `산하조직`, `본부장`, `팀장`, `소속팀원`, `직속보고`, `주관부서`, `총괄PM`, `참여인력`, `파견협업`, `예산승인권자`, `사전합의및감사`, `선후행의존` | 관계 명칭이 제각각으로 추출되는 현상을 방지 |
| **엔티티 정규화 (`canonicalize_entity`)** | `'클운팀'` → `'클라우드운영팀'`, `'김철수'` → `'김철수 수석'`, `'P-01'` → `'P-01: 클라우드 네이티브 마이그레이션'` | 서로 다른 표기로 등장한 동일 개체를 하나의 노드로 통합 (Entity Resolution) |

먼저 이 방식으로 텍스트에서 직접 지식 그래프를 구축하고 다중 홉 탐색이 어떻게 이루어지는지 원리를 확인한 뒤, **Part 8에서 대규모 문서에 적용하는 Microsoft GraphRAG**로 넘어가겠습니다.

In [ ]:
with open("data/graph/org_relationships.txt", "r", encoding="utf-8") as f:
    org_text = f.read()
print(org_text[:700])

In [ ]:
from rag.graph.triplet_extractor import extract_triplets, EntityType, RelationType

kg = extract_triplets(org_text, llm=llm)   # 도메인 온톨로지 제약(EntityType, RelationType) 기반 추출
print(f"✅ 온톨로지 제약 트리플렛 {len(kg.triplets)}개 추출 완료\n")
for t in kg.triplets[:12]:
    detail_str = f" ({t.detail})" if t.detail else ""
    print(f"  [{t.subject} ({t.subject_type})] ──({t.relation}{detail_str})──▶ [{t.object} ({t.object_type})]")

In [ ]:
from rag.graph.networkx_graph import build_graph_from_triplets, visualize_pyvis, multi_hop_search
from IPython.display import IFrame

G = build_graph_from_triplets(kg.triplets)
print(f"노드 {G.number_of_nodes()}개, 엣지 {G.number_of_edges()}개")

os.makedirs("artifacts", exist_ok=True)
html_path = visualize_pyvis(G, output_html="artifacts/knowledge_graph.html")
print(f"🌐 인터랙티브 그래프 저장: {html_path} (브라우저로 열면 노드를 드래그할 수 있습니다)")

# IFrame 경로는 노트북 파일 위치(notebooks/) 기준이므로 상위 폴더로 올라갑니다.
IFrame("../" + html_path, width="100%", height=520)

In [ ]:
for hops in (1, 2):
    edges, visited = multi_hop_search(G, seed_entities=["김철수 수석"], max_hops=hops)
    print(f"\n🔍 {hops}-Hop 탐색: 방문 노드 {len(visited)}개, 관계 {len(edges)}개")
    for u, v, d in edges[:10]:
        print(f"   [{u}] ──({d['relation']})──▶ [{v}]")

💡 **1-Hop 탐색과 2-Hop 탐색의 차이점:**

1. **1-Hop 탐색 (직접 연결된 관계만 조회)**: 시작점인 `김철수 수석`과 직접 연결된 노드(`클라우드운영팀`, `P-01 프로젝트`)까지만 확인하므로, 해당 프로젝트의 최종 예산 승인권자인 `박영희 전무`까지는 도달하지 못합니다.
2. **2-Hop 탐색 (한 단계 더 건너간 관계까지 조회)**: `클라우드운영팀`의 상위 조직인 `클라우드사업본부`와, `P-01 프로젝트`의 예산 승인권자인 `박영희 전무`까지 연결 고리를 따라 탐색하여 **복합 질문에 필요한 전체 관계망을 찾아냅니다.**
3. **에이전트 도구 설계 시 고려사항**: 탐색 깊이(`max_hops`)를 무조건 크게 설정하면 연결된 노드가 급격히 늘어나 불필요한 정보까지 LLM 컨텍스트에 포함될 수 있습니다. 따라서 에이전트 도구에서는 `max_hops`를 인자로 열어 두어, 질문이 단순한지 복합적인지에 따라 에이전트가 탐색 범위를 조절하도록 설계합니다.

> 💡 **실무 환경에서의 엔티티 정규화(Entity Resolution)**:
> 이번 실습 데이터는 그래프 탐색 원리를 직관적으로 확인할 수 있도록 구성된 예제입니다. 실제 기업 문서에서는 같은 인물이나 부서가 다양한 호칭(`김철수 수석` / `김철수 팀장`)으로 등장하므로, 앞단에서 동의어 사전이나 유사도 클러스터링을 통해 동일 개체를 하나로 묶어주는 전처리가 중요합니다.

---
## 🌐 Part 8. Microsoft GraphRAG: 세부 관계 탐색(Local)과 전사 요약(Global)의 통합

### 왜 실무에서는 Microsoft GraphRAG를 활용할까요?

Part 7에서는 직접 정의한 온톨로지 스키마로 트리플렛을 추출하고 그래프를 탐색하는 기본 원리를 살펴보았습니다.
하지만 실제 기업의 문서는 정형화된 조직도뿐만 아니라 **아키텍처 위원회 회의록, 장애 사후분석 보고서(Post-mortem), 보안 감사 리포트**처럼 길고 복잡한 서술형 문서들로 이루어져 있습니다. 또한 사용자들이 던지는 질문도 크게 두 가지 성격으로 나뉩니다.

1. **특정 개체 중심의 미시적 질문 (`Local Search`)**: *"클라우드운영팀 김철수 수석의 역할과 보고 라인, 담당 프로젝트는 무엇인가?"*
2. **문서 전체를 아우르는 거시적 질문 (`Global Search`)**: *"전사 전략 프로젝트들의 공통 목표와 핵심 리스크를 종합해 줘."*

Microsoft가 제안한 **GraphRAG**는 하나의 인덱싱 파이프라인을 통해 **개체 및 관계 그래프(`entities.parquet`, `relationships.parquet`)**와, 밀접하게 연결된 노드 군집별로 미리 요약해 둔 **계층적 커뮤니티 리포트(`community_reports.parquet`)**를 함께 생성합니다. 덕분에 세부적인 관계 추적(`Local Search`)과 전체적인 흐름 요약(`Global Search`)을 하나의 프레임워크에서 모두 처리할 수 있습니다.

---

### 🏗️ Microsoft GraphRAG 인덱싱 파이프라인과 산출물 구조

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    A["사내 통합 문서<br/>(조직·회의록·감사)"] --> B["1. 텍스트 분할<br/>(text_units.parquet)"]
    B --> C["2. 개체·관계 추출<br/>(entities · relationships)"]
    C --> D["3. Leiden 군집화<br/>(communities.parquet)"]
    D --> E["4. 군집별 LLM 요약<br/>(community_reports.parquet)"]
    C --> F["5. 벡터 임베딩 색인<br/>(LanceDB)"]
    E --> F

    style C fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style E fill:#fff8e1,stroke:#f9a825,stroke-width:1.5px
    style F fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

### 📂 `app/database/graphrag/output/` 내 주요 테이블 구성
- `entities.parquet`: 인물, 부서, 전략 프로젝트(P-01\~P-06), 주요 이슈 등 추출된 개체 목록과 요약 설명
- `relationships.parquet`: 개체 간의 연결 관계(`[source] ──▶ [target]`), 관계 설명 및 연결 강도(weight)
- `community_reports.parquet`: Leiden 알고리즘으로 묶인 각 커뮤니티(하위 그룹)의 핵심 내용을 LLM이 미리 정리해 둔 요약 보고서
- `text_units.parquet`: 개체와 관계가 추출된 원본 텍스트 청크

---

### 🧠 Microsoft GraphRAG의 기본 검색 방식: Local Search vs Global Search

GraphRAG는 인덱스를 만든 뒤 두 가지 검색 방식으로 **질문에 대한 답변을 직접 생성**합니다.

| 검색 방식 | 참조하는 데이터 | 동작 방식 | 적합한 질문 유형 |
|:---|:---|:---|:---|
| **Local Search** | `entities` + `relationships` + `community_reports` + `text_units` | 질문 임베딩과 가까운 엔티티를 찾고, 연결된 관계·커뮤니티 리포트·원문 조각을 토큰 상한까지 채워 맥락을 만든 뒤 **LLM이 답변을 작성** | 특정 인물·부서·프로젝트의 결재선, 협업 관계, 담당 업무 |
| **Global Search** | `community_reports` | 리포트 묶음마다 LLM이 질문 관련 핵심 포인트와 중요도 점수를 뽑고(**Map**, 병렬 호출), 점수가 높은 포인트를 모아 **LLM이 최종 답변을 작성**(**Reduce**) | 전사 공통 목표, 핵심 리스크, 동향 요약 |

먼저 GraphRAG가 제공하는 CLI로 두 방식을 그대로 실행해 보고, 이 방식을 에이전트의 검색 도구로 쓸 수 있을지 따져 보겠습니다.

In [ ]:
import pandas as pd

out_dir = "app/database/graphrag/output"
entities = pd.read_parquet(f"{out_dir}/entities.parquet")
reports = pd.read_parquet(f"{out_dir}/community_reports.parquet")
print(f"엔티티 {len(entities)}개, 커뮤니티 리포트 {len(reports)}개\n")
for _, row in reports.head(3).iterrows():
    print(f"[커뮤니티 {row['community']}] {row['title']}\n   {row['summary'][:150]}...\n")

### ▶️ GraphRAG CLI로 Local / Global 검색 실행하기

GraphRAG를 설치하면 `graphrag` 명령이 함께 설치됩니다. 인덱스가 있는 프로젝트 폴더(`--root`)와 검색 방식(`--method`)을 지정하면 **검색부터 답변 생성까지** 한 번에 수행합니다.

```bash
graphrag query "질문" --root app/database/graphrag --method local   # 또는 global
```

노트북에서는 `subprocess`로 같은 명령을 실행하고, **걸린 시간**을 함께 측정합니다.

In [ ]:
import subprocess
import time

def graphrag_cli(query: str, method: str) -> str:
    """GraphRAG CLI로 검색과 답변 생성을 한 번에 실행하고, 걸린 시간을 출력합니다."""
    start = time.time()
    proc = subprocess.run(
        [sys.executable, "-m", "graphrag", "query", query, "--root", "app/database/graphrag", "--method", method],
        capture_output=True, text=True,
    )
    print(f"⏱️ [{method}] {time.time() - start:.1f}초\n")
    return proc.stdout if proc.returncode == 0 else proc.stderr[-2000:]

local_answer = graphrag_cli("클라우드운영팀 김철수 수석의 역할과 보고 라인, 담당 프로젝트는?", method="local")
print(local_answer)

In [ ]:
global_answer = graphrag_cli("전사 프로젝트들의 공통 목표와 핵심 리스크를 요약해줘.", method="global")
print(global_answer)

### 🤔 이 방식을 에이전트의 검색 도구로 그대로 쓸 수 있을까?

답변 품질은 좋지만, CLI/엔진 일체형 검색을 에이전트의 도구로 그대로 연결하기에는 세 가지 한계가 있습니다.

1. **속도가 느립니다.** Local은 수십 초, Global은 1분 안팎이 걸립니다. 도구 내부에서 LLM이 답변까지 생성하기 때문이며, 특히 Global은 리포트 묶음마다 Map 호출을 수행한 뒤 Reduce 호출을 한 번 더 거칩니다.
2. **답변 생성이 중복됩니다.** 도구가 만든 요약 답변을 받아 에이전트 LLM이 다시 최종 답변을 작성하므로 비용이 이중으로 들고, 중간 요약 과정에서 정확한 수치나 조건이 누락될 수 있습니다.
3. **원문 근거를 역추적하기 어렵습니다.** 결과 속 `[Data: Entities (8); Relationships (12)]` 표기만으로는 에이전트가 실제 원문 구절을 다시 열어 다른 도구(SQL·규정 검색) 결과와 교차 검증하기 어렵습니다.

**답변 생성만 빼고 Local Search의 맥락 묶음을 통째로 반환하면 어떨까요?** 속도는 빨라지지만, Local Search 맥락 생성기는 토큰 상한까지 관계·리포트·원문을 한 번에 채워 넣도록 설계되어 있어 도구 호출 1회에 수만 자가 반환됩니다.

### 🧭 해결: 인덱스는 GraphRAG 산출물 그대로, 탐색은 5종 도구로 단계적 수행

GraphRAG의 핵심 자산은 무거운 검색 루프보다 **사전 구축된 구조화 인덱스**(`entities`, `relationships`, `community_reports`, `text_units`, `lancedb` 임베딩)입니다. 따라서 인덱스는 그대로 활용하되, 에이전트가 질문 성격에 맞춰 필요한 만큼만 따라가도록 **가벼운 탐색 도구 5종**으로 분리합니다.

| 도구 | 하는 일 | 대응하는 GraphRAG 개념 |
|:---|:---|:---|
| `graph_find_entities` | 질문 속 인물·부서·프로젝트를 그래프 엔티티(`E번호`)로 매핑 (탐색 시작점) | Local의 엔티티 매핑 |
| `graph_neighbors` | 특정 엔티티의 1~2홉 관계(`R번호`)를 질문 유사도 순으로 조회 | Local의 주변 관계 수집 |
| `graph_find_path` | 두 엔티티 사이의 최단 경로 조회 (보고선, 선후행 의존 체인) | 그래프 경로 탐색 |
| `graph_search_communities` | 사전 요약된 커뮤니티 리포트(`C번호`)를 임베딩 유사도로 즉시 검색 | Global의 경량 대안 (Map·Reduce 생략) |
| `graph_read` | `E·R·C·T` ID를 지정해 상세 설명·전체 리포트·원문 조각(`T번호`) 열람 | 점진적 근거 확인 (Progressive Disclosure) |

설계 원칙은 세 가지입니다.
- **도구 내부에서 LLM을 호출하지 않습니다.** 질문 임베딩 1회와 메모리 그래프 탐색만 수행하므로 도구 1회 호출이 1초 이내에 끝납니다.
- **목록은 짧게 반환하고 고유 ID(`E`·`R`·`C`·`T`)를 부여합니다.**
- **세부 수치와 원문은 필요할 때만 `graph_read`로 펼쳐 읽습니다.**

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 760, 'padding': 16}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    Q["사용자 질문"]

    subgraph LOCAL_TRACK["미시적 관계 · 경로 추적 (특정 부서·인물·프로젝트 질문)"]
        direction LR
        FE["1. graph_find_entities<br/>이름 → 엔티티 ID(E22) 매핑"]
        NB["2A. graph_neighbors<br/>주변 1~2홉 관계(R번호) 조회"]
        FP["2B. graph_find_path<br/>두 개체 간 최단 경로(P-04↔P-01)"]
        FE --> NB
        FE --> FP
    end

    subgraph GLOBAL_TRACK["거시적 동향 파악 (전사 목표·핵심 리스크 질문)"]
        direction LR
        SC["1. graph_search_communities<br/>사전 요약 리포트(C번호) 벡터 검색<br/>(Map·Reduce LLM 호출 없음)"]
    end

    RD["3. graph_read(ids=[...])<br/>필요한 E·R·C·T ID만 골라<br/>상세 리포트 · 근거 원문(T27) 정밀 열람"]

    Q -->|"개별 대상 · 결재선"| FE
    Q -->|"전사 개요 · 리스크"| SC
    NB -->|"근거 T번호 확인"| RD
    FP -->|"근거 T번호 확인"| RD
    SC -->|"상세 C번호 확인"| RD

    style LOCAL_TRACK fill:#e3f2fd,stroke:#1976d2,stroke-width:1.5px
    style GLOBAL_TRACK fill:#fff8e1,stroke:#f9a825,stroke-width:1.5px
    style RD fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
```

> 💡 **DRIFT Search와의 관계**: Microsoft는 Global로 개요를 잡고 후속 질문마다 Local Search를 반복하는 **DRIFT Search**도 제공합니다. 위 5종 도구 구조는 DRIFT의 *"넓게 조망하고 필요한 곳만 좁혀 들어가는"* 탐색 전략을 **에이전트 자신의 ReAct 도구 호출 루프**로 옮겨 경량화한 형태입니다.

### 🔍 탐색 도구 5종 직접 호출해 보기

예시 질문: *"AgenticAI팀이 총괄하는 프로젝트의 예산과 승인권자, 그리고 의존하는 프로젝트는?"*

에이전트가 하게 될 탐색을 먼저 손으로 따라가 봅니다. 각 호출의 **시간**과 **결과 크기**를 위의 CLI 결과와 비교해 보세요.

In [ ]:
from rag.graph.graph_explorer import (
    graph_find_entities,
    graph_find_path,
    graph_neighbors,
    graph_read,
    graph_search_communities,
)

def run(tool, args):
    start = time.time()
    out = tool.invoke(args)
    print(f"⏱️ {tool.name}: {time.time() - start:.2f}초 · {len(out):,}자\n\n{out}")

# 1) 질문 속 이름을 그래프 엔티티로 연결 (그래프의 엔티티 이름은 영문)
run(graph_find_entities, {"query": "AgenticAI팀"})

In [ ]:
# 2) 팀에서 뻗어 나가는 관계 중 '총괄 프로젝트'와 가까운 것부터
run(graph_neighbors, {"entity": "AGENTIC AI TEAM", "query": "총괄하는 프로젝트"})

In [ ]:
# 3) 찾은 프로젝트(P-02)의 예산·승인권자·의존 관계
run(graph_neighbors, {"entity": "P-02", "query": "예산 승인권자 의존 프로젝트", "top_k": 8})

In [ ]:
# 4) 수치는 원문으로 확인: 위 결과 끝에 안내된 근거 원문 ID를 읽기
run(graph_read, {"ids": ["T27"]})

In [ ]:
# 5) 두 프로젝트는 어떻게 이어져 있을까? (의존 체인)
run(graph_find_path, {"source": "P-04", "target": "P-01"})

In [ ]:
# 6) 전사 개요: Global Search 대신 커뮤니티 리포트 요약을 바로 검색 (LLM 호출 없음)
run(graph_search_communities, {"query": "전사 프로젝트들의 공통 목표와 핵심 리스크", "level": 0})

### 🤖 에이전트에 연결하기

이제 어떤 도구를 어떤 순서로 부를지를 에이전트에게 맡깁니다. 에이전트의 **도구 호출 순서**와 **도구 결과의 총량**을 확인해 보세요.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage
from rag.graph.graph_explorer import graph_explorer_tools

graph_agent = create_agent(
    model=llm,
    tools=graph_explorer_tools,
    system_prompt="당신은 사내 지식 그래프 탐색 에이전트입니다. 도구로 근거를 찾아 한국어로 답하고, 근거가 된 수치를 함께 쓰세요.",
)

question = "AgenticAI팀이 총괄하는 프로젝트의 예산과 승인권자, 그리고 이 프로젝트가 의존하는 다른 프로젝트는?"
start = time.time()
result = graph_agent.invoke({"messages": [("user", question)]})
elapsed = time.time() - start

for message in result["messages"]:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            print(f"🔧 {call['name']}({call['args']})")
tool_chars = sum(len(normalize_content(m.content)) for m in result["messages"] if isinstance(m, ToolMessage))
print(f"\n⏱️ 전체 {elapsed:.1f}초 · 도구 결과 합계 {tool_chars:,}자\n")
print(normalize_content(result["messages"][-1].content))

### 📊 답변 생성형 검색 vs 단계적 탐색 도구

| 비교 항목 | GraphRAG Local / Global (CLI) | 탐색 도구 5종 + 에이전트 |
|:---|:---|:---|
| **도구 1회 시간** | Local 수십 초, Global 1분 안팎 | 1초 이내 |
| **도구 내부 LLM 호출** | 있음 (Local 1회, Global은 Map 여러 회 + Reduce) | 없음 (질문 임베딩 1회) |
| **반환 결과** | 완성된 답변 (근거를 다시 열어 볼 수 없음) | 짧은 검색 결과 + ID (근거를 다시 열어 볼 수 있음) |
| **탐색 방식** | 질문 하나로 맥락을 한 번에 구성 | 에이전트가 필요한 관계·원문만 단계적으로 조회 |
| **비용 구조** | 질문마다 고정적으로 큼 | 질문이 요구하는 탐색 깊이(홉 수)에 비례 |

> ⚠️ **트레이드오프**: 단계적 탐색은 도구를 여러 번 부르므로 LLM 왕복이 늘어납니다. 문서가 적을 때는 큰 맥락을 한 번에 넣는 방식이 더 빠를 수 있습니다. 하지만 문서가 늘어날수록 한 번에 담을 수 있는 비율은 줄어드는 반면, 단계적 탐색의 비용은 문서 규모가 아니라 질문의 탐색 깊이에 따라 정해집니다.

---
## 🎓 엔터프라이즈 인덱싱 기법 종합 비교

| 비교 항목 | 하이브리드 벡터 검색 (Part 1\~6) | 온톨로지 지식 그래프 기초 (Part 7) | Microsoft GraphRAG (Part 8) |
| :--- | :--- | :--- | :--- |
| **핵심 기법** | 표준 Markdown 정규화 + 문맥 헤더(`search`/`read`) + Dense/BM25/Window | 도메인 온톨로지 스키마(`EntityType`/`RelationType`) + NetworkX | Leiden 계층 군집화 + 커뮤니티 리포트 + 단계적 그래프 탐색 도구 |
| **적합한 질문** | 조항·절차·가이드라인 등 문서 내 구체적 지침 검색 | 개체 간 다단계 연결(Multi-Hop) 탐색 원리 학습 | 복합 결재선·협업 관계·의존 체인 추적 및 전사 동향 개괄 |
| **연동되는 에이전트 도구** | `search_company_policy`, `search_aws_guides` | *(원리 실습용 — 실제 에이전트 도구는 GraphRAG 사용)* | `graph_find_entities`, `graph_neighbors`, `graph_find_path`, `graph_search_communities`, `graph_read` |

> **🌟 전체 과정에서의 위치**:
> 엔터프라이즈 Agentic RAG 시스템은 데이터의 성격에 맞춰 서로 다른 인덱스와 검색 도구를 조합합니다.
> - **규정 및 기술 가이드 문서 (`rag/vector`)** ➔ `search_company_policy`, `search_aws_guides`
> - **조직·결재선·프로젝트 관계망 (`rag/graph`)** ➔ `graph_*` 탐색 도구 5종 (엔티티 연결 → 관계·경로 탐색 → 원문 확인)
> - **예산·인사·인프라 현황 등 정형 DB (`rag/sql`)** ➔ `search_enterprise_erp` (다음 `4_Text2SQL.ipynb`에서 상세 실습)
> 
> 이렇게 각 데이터 특성에 맞게 인덱싱해 두면, **에이전트가 사용자의 질문을 분석해 가장 알맞은 검색 도구를 스스로 선택하고 조합하여 답변**할 수 있습니다.

---
## 🏁 Part 9. 전체 요약 및 운영 데이터베이스 검증

이번 실습에서 다룬 인덱싱 및 검색 파이프라인의 전체 단계를 정리하면 다음과 같습니다.

| 단계 | 핵심 기법 | 해결한 문제 | 주요 함수 및 클래스 |
|:---:|:---|:---|:---|
| **정규화** | 북마크 및 반복 패턴 기반 마크다운 변환 | 계층 태그가 없는 PDF의 문서 구조 복원 (포맷 파편화 해결) | `pdf_to_markdown()`, `to_markdown()` |
| **정규화** | 텍스트 보존율 및 헤더 분포 검수 | 변환 과정에서 텍스트가 누락되거나 계층이 깨지는 문제 검증 | `measure_text_coverage()` |
| **청킹** | 검색용/읽기용 문맥 헤더 및 경로(Breadcrumb) 주입 | 청크 분할 시 출처 유실 방지 및 긴 목차로 인한 임베딩 희석 방지 | `ContextualHeader` |
| **청킹** | 도메인 메타데이터 태깅 및 통합 인덱싱 | 동일 양식의 여러 문서에서 같은 소제목이 반복될 때 영역 구분 | `index_documents()`, `aws_metadata()` |
| **검색** | 도메인 용어 사전 정규화 | 사용자의 질문 용어(약어)와 문서 내 공식 명칭의 불일치 해결 | `DomainDictionary` |
| **검색** | BM25 + Dense 하이브리드 검색 (RRF) | 의미 유사도 검색과 정확한 키워드/조항 번호 매칭을 동시에 달성 | `HybridRetriever`, `reciprocal_rank_fusion()` |
| **복원** | 인접 청크 윈도우 확장 | 청크 경계에서 잘린 표와 단서 조항을 앞뒤 청크 결합으로 복원 | `ContextualWindow` |
| **그래프** | 도메인 온톨로지 제약 기반 트리플렛 추출 | 관계 어휘와 개체 명칭이 제각각 추출되어 그래프가 끊어지는 문제 방지 | `extract_triplets()` |
| **그래프** | Microsoft GraphRAG 인덱스 + 단계적 탐색 도구 | 답변 생성형 검색의 지연과 맥락 비대 없이 필요한 관계·원문만 조회 | `graph_find_entities()` 외 4종 |

> 💡 **확장 포인트:** **새로운 형식이나 양식의 문서가 추가될 때는 앞단의 정규화 설정(Preset)만 추가하면 됩니다.** 표준 Markdown(`H1~H4`)으로 변환된 이후의 청킹, 임베딩, 하이브리드 검색, 에이전트 도구 코드는 수정 없이 그대로 사용할 수 있습니다.

### 🧪 운영 데이터베이스(`app/database/`) 통합 검증 및 제로 코드 확장

실습 환경의 `app/database/`에는 이번 노트북에서 다룬 파이프라인(`app/mcp/build_database.py`)으로 구축된 4종 데이터베이스(`enterprise_policy_store` 239청크, `aws_waf_guides_store` 4,641청크, `knowledge_graph_nodelink.json`, `graphrag/`)가 사전 빌드되어 있습니다.
- **통합 검색 검증**: 아래 셀을 실행하여 `app/mcp/test_database.py`로 4대 데이터베이스 검색이 모두 정상 동작하는지 확인해 보세요.
- **신규 규정 제로 코드 추가 (`--extra-docs`)**: 새로운 규정 문서(`data/new_documents/2026_복리후생지원규정.md`)가 제정되었을 때도 코드 수정 없이 다음 명령 한 줄로 즉시 증분 인덱싱(239 ➔ 272청크) 및 원상 복구(롤백)할 수 있습니다.
  ```bash
  # 신규 복리후생 규정 추가 인덱싱 (239 ➔ 272청크)
  python app/mcp/build_database.py --targets policy --extra-docs data/new_documents

  # 기본 5대 규정 초기 상태로 롤백 (239청크)
  python app/mcp/build_database.py --targets policy
  ```

In [ ]:
from app.mcp.test_database import (
    test_1_policy_vector_db,
    test_2_aws_vector_db,
    test_3_knowledge_graph_db,
    test_4_graphrag_db,
)

test_1_policy_vector_db()
test_2_aws_vector_db()
test_3_knowledge_graph_db()
test_4_graphrag_db()